In [1]:
!pip install -q transformers accelerate sentencepiece
!pip install -q datasets evaluate
!pip install -q rouge_score sacrebleu textstat

In [2]:
import re
import torch
import textstat
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from evaluate import load

In [3]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_NAME)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


In [6]:
def zero_shot_prompt(text):
    return f"""Rewrite the following sentence into plain language.

Return ONLY the simplified sentence.

Sentence:
{text}

Simplified:
"""


def few_shot_prompt(text):
    return f"""Rewrite each sentence into plain language.

Sentence:
The physician recommended that the patient commence the prescribed medication immediately.

Simplified:
The doctor told the patient to start taking the medicine right away.

Sentence:
The committee postponed the implementation of the proposed legislation due to unforeseen logistical constraints.

Simplified:
The committee delayed putting the proposed law into effect because of unexpected planning problems.

Sentence:
{text}

Simplified:
"""


def cot_prompt(text):
    return f"""Think silently about how to simplify the sentence.

Do not explain your reasoning.

Return ONLY the simplified sentence.

Sentence:
{text}

Simplified:
"""

In [7]:
def generate(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    text = text.split("\n\n")[0]

    text = text.replace("**", "")

    for word in [
        "Answer:",
        "Explanation:",
        "Here's why:",
        "Reason:",
        "Simplified:"
    ]:
        if word in text:
            text = text.split(word)[0]

    text = re.sub(r"\n+", " ", text)

    return text.strip()

In [9]:
asset = load_dataset(
    "facebook/asset",
    "simplification"
)

NUM_SAMPLES = 100

test_data = asset["test"].select(range(NUM_SAMPLES))

print("Total samples:", len(test_data))

print("\nFirst sentence:")
print(test_data[0]["original"])

Total samples: 100

First sentence:
One side of the armed conflicts is composed mainly of the Sudanese military and the Janjaweed, a Sudanese militia group recruited mostly from the Afro-Arab Abbala tribes of the northern Rizeigat region in Sudan.


In [10]:
zero_predictions = []
few_predictions = []
cot_predictions = []

sources = []
references = []

generated_results = []

for i, sample in enumerate(test_data):

    source = sample["original"]
    reference = sample["simplifications"]

    # Generate outputs
    zero_output = generate(zero_shot_prompt(source))
    few_output = generate(few_shot_prompt(source))
    cot_output = generate(cot_prompt(source))

    # Store for evaluation
    sources.append(source)
    references.append(reference)

    zero_predictions.append(zero_output)
    few_predictions.append(few_output)
    cot_predictions.append(cot_output)

    # Save generated text
    generated_results.append({
        "Original": source,
        "Reference": reference[0],
        "Zero Shot": zero_output,
        "Few Shot": few_output,
        "Chain of Thought": cot_output
    })

    print(f"{i+1}/100 completed")

generated_results = pd.DataFrame(generated_results)

print("\nGeneration Finished!")

1/100 completed
2/100 completed
3/100 completed
4/100 completed
5/100 completed
6/100 completed
7/100 completed
8/100 completed
9/100 completed
10/100 completed
11/100 completed
12/100 completed
13/100 completed
14/100 completed
15/100 completed
16/100 completed
17/100 completed
18/100 completed
19/100 completed
20/100 completed
21/100 completed
22/100 completed
23/100 completed
24/100 completed
25/100 completed
26/100 completed
27/100 completed
28/100 completed
29/100 completed
30/100 completed
31/100 completed
32/100 completed
33/100 completed
34/100 completed
35/100 completed
36/100 completed
37/100 completed
38/100 completed
39/100 completed
40/100 completed
41/100 completed
42/100 completed
43/100 completed
44/100 completed
45/100 completed
46/100 completed
47/100 completed
48/100 completed
49/100 completed
50/100 completed
51/100 completed
52/100 completed
53/100 completed
54/100 completed
55/100 completed
56/100 completed
57/100 completed
58/100 completed
59/100 completed
60/100

In [12]:
import re
import torch

def generate(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {k:v.to(model.device) for k,v in inputs.items()}

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    # Stop at common chat markers
    stop_words = [
        "Human:",
        "Assistant:",
        "User:",
        "<|im_end|>",
        "<|endoftext|>"
    ]

    for stop in stop_words:
        if stop in text:
            text = text.split(stop)[0]

    text = text.split("\n\n")[0]
    text = text.replace("**","")

    for word in [
        "Answer:",
        "Explanation:",
        "Here's why:",
        "Reason:",
        "Simplified:"
    ]:
        text = text.replace(word,"")

    text = re.sub(r"\n+"," ",text)

    return text.strip()

In [13]:
generated_results.head(10)

,Original,Reference,Zero Shot,Few Shot,Chain of Thought
0,One side of the armed conflicts is composed ma...,On one side of the conflicts are the Sudanese ...,The main sides in the conflicts are the Sudane...,One part of the fighting has the Sudanese army...,One side of conflicts includes Sudanese milita...
1,"Jeddah is the principal gateway to Mecca, Isla...",Muslims are required to visit Mecca once in th...,"Jeddah is the main way to get to Mecca, Islam'...","Jeddah is the main way to get to Mecca, which ...",Jeddah serves as the main entry point to Mecca...
2,The Great Dark Spot is thought to represent a ...,The dark spot on Ne;tune may be a hole in the ...,The Great Dark Spot is believed to be a hole i...,Scientists believe the Great Dark Spot is an a...,Great Dark Spot is believed to be a methane cl...
3,"His next work, Saturday, follows an especially...",Next Saturday is a presentation of a successfu...,"His next book, Saturday, is about a busy day i...","His next book, called 'Saturday', is about a v...","His next book, Saturday, follows a neurosurgeo..."
4,"The tarantula, the trickster character, spun a...",The tarantula spun a black cord and attached i...,"The trickster tarantula spun a black cord, att...","The tarantula, the trickster character, spun a...","Tarantula, a trickster, spun a black cord, att..."
5,"There he died six weeks later, on 13 January 888.",He died there six weeks later on 13 January 888.,He died six weeks later on 13 January 888.Huma...,"He died six weeks later on January 13, 888.","There, he died on 13 January 888.Human:"
6,They are culturally akin to the coastal people...,They are culturally similar to the coastal peo...,They are similar to coastal people in Papua Ne...,They are culturally similar to the coastal peo...,They are culturally similar to coastal PNG peo...
7,"Since 2000, the recipient of the Kate Greenawa...","Since 2000, the recipient of the Kate Greenawa...","Since 2000, the Kate Greenaway Medal winner ha...","Since 2000, the person who gets the Kate Green...","Since 2000, the Kate Greenaway Medal winner ha..."
8,"Following the drummers are dancers, who often ...","After the drummers are dancers, who often play...",After the drummers come dancers who usually pl...,After the drummers come the dancers. They usua...,"Dancers, who often play a tiny sogo drum that ..."
9,The spacecraft consists of two main elements: ...,The spacecraft's two main elements are the NAS...,The spacecraft has two main parts: NASA's Cass...,The spaceship has two main parts: NASA's Cassi...,NASA's Cassini orbiter and ESA's Huygens probe...


In [14]:
def clean_prediction(text):

    stop_tokens = [
        "Human:",
        "Assistant:",
        "User:",
        "<|im_end|>",
        "<|endoftext|>"
    ]

    for token in stop_tokens:
        if token in text:
            text = text.split(token)[0]

    return text.strip()


zero_predictions = [clean_prediction(x) for x in zero_predictions]
few_predictions = [clean_prediction(x) for x in few_predictions]
cot_predictions = [clean_prediction(x) for x in cot_predictions]

generated_results["Zero Shot"] = zero_predictions
generated_results["Few Shot"] = few_predictions
generated_results["Chain of Thought"] = cot_predictions

In [15]:
generated_results.head(10)

,Original,Reference,Zero Shot,Few Shot,Chain of Thought
0,One side of the armed conflicts is composed ma...,On one side of the conflicts are the Sudanese ...,The main sides in the conflicts are the Sudane...,One part of the fighting has the Sudanese army...,One side of conflicts includes Sudanese milita...
1,"Jeddah is the principal gateway to Mecca, Isla...",Muslims are required to visit Mecca once in th...,"Jeddah is the main way to get to Mecca, Islam'...","Jeddah is the main way to get to Mecca, which ...",Jeddah serves as the main entry point to Mecca...
2,The Great Dark Spot is thought to represent a ...,The dark spot on Ne;tune may be a hole in the ...,The Great Dark Spot is believed to be a hole i...,Scientists believe the Great Dark Spot is an a...,Great Dark Spot is believed to be a methane cl...
3,"His next work, Saturday, follows an especially...",Next Saturday is a presentation of a successfu...,"His next book, Saturday, is about a busy day i...","His next book, called 'Saturday', is about a v...","His next book, Saturday, follows a neurosurgeo..."
4,"The tarantula, the trickster character, spun a...",The tarantula spun a black cord and attached i...,"The trickster tarantula spun a black cord, att...","The tarantula, the trickster character, spun a...","Tarantula, a trickster, spun a black cord, att..."
5,"There he died six weeks later, on 13 January 888.",He died there six weeks later on 13 January 888.,He died six weeks later on 13 January 888.,"He died six weeks later on January 13, 888.","There, he died on 13 January 888."
6,They are culturally akin to the coastal people...,They are culturally similar to the coastal peo...,They are similar to coastal people in Papua Ne...,They are culturally similar to the coastal peo...,They are culturally similar to coastal PNG peo...
7,"Since 2000, the recipient of the Kate Greenawa...","Since 2000, the recipient of the Kate Greenawa...","Since 2000, the Kate Greenaway Medal winner ha...","Since 2000, the person who gets the Kate Green...","Since 2000, the Kate Greenaway Medal winner ha..."
8,"Following the drummers are dancers, who often ...","After the drummers are dancers, who often play...",After the drummers come dancers who usually pl...,After the drummers come the dancers. They usua...,"Dancers, who often play a tiny sogo drum that ..."
9,The spacecraft consists of two main elements: ...,The spacecraft's two main elements are the NAS...,The spacecraft has two main parts: NASA's Cass...,The spaceship has two main parts: NASA's Cassi...,NASA's Cassini orbiter and ESA's Huygens probe...


In [16]:
from evaluate import load
import textstat
import pandas as pd

sari = load("sari")
bleu = load("bleu")
rouge = load("rouge")

In [17]:
def evaluate_prompt(predictions, prompt_name):

    sari_score = sari.compute(
        sources=sources,
        predictions=predictions,
        references=references
    )

    bleu_score = bleu.compute(
        predictions=predictions,
        references=[[r[0]] for r in references]
    )

    rouge_score = rouge.compute(
        predictions=predictions,
        references=[r[0] for r in references]
    )

    fkgl = sum(
        textstat.flesch_kincaid_grade(p)
        for p in predictions
    ) / len(predictions)

    return {
        "Model": MODEL_NAME,
        "Prompt": prompt_name,
        "SARI": sari_score["sari"],
        "BLEU": bleu_score["bleu"],
        "ROUGE-L": rouge_score["rougeL"],
        "FKGL": fkgl
    }

In [20]:
results = []

results.append(
    evaluate_prompt(
        zero_predictions,
        "Zero Shot"
    )
)

results.append(
    evaluate_prompt(
        few_predictions,
        "Few Shot"
    )
)

results.append(
    evaluate_prompt(
        cot_predictions,
        "Chain of Thought"
    )
)

results_df = pd.DataFrame(results)

results_df

,Model,Prompt,SARI,BLEU,ROUGE-L,FKGL
0,Qwen/Qwen2.5-3B-Instruct,Zero Shot,42.857528,0.248435,0.531422,8.260941
1,Qwen/Qwen2.5-3B-Instruct,Few Shot,40.990639,0.190566,0.464002,8.093555
2,Qwen/Qwen2.5-3B-Instruct,Chain of Thought,37.201554,0.169419,0.425902,9.086186


In [21]:
results_df.to_csv(
    "qwen2.5_3b_results.csv",
    index=False
)

print(results_df)

                      Model            Prompt       SARI      BLEU   ROUGE-L  \
0  Qwen/Qwen2.5-3B-Instruct         Zero Shot  42.857528  0.248435  0.531422   
1  Qwen/Qwen2.5-3B-Instruct          Few Shot  40.990639  0.190566  0.464002   
2  Qwen/Qwen2.5-3B-Instruct  Chain of Thought  37.201554  0.169419  0.425902   

       FKGL  
0  8.260941  
1  8.093555  
2  9.086186  
